In Previous Project we maped healthcare code for Billing Purpose  But Billing is sensitive and needs certified coder review, but  Now  we Map the code for  reporting/research


Clinical Note
↓
Tokenization + Normalization
↓
NER, find entities
↓
Context + Time understanding
↓
Classification, predict concepts
↓
Map to ICD/CPT/SNOMED-style candidates
↓
Confidence + human review
↓
Final coded output for reporting/research

Install libraries

In [1]:
!pip install pandas transformers torch python-dateutil



Import libraries

In [2]:
import pandas as pd
import re
import torch
import numpy as np

from transformers import pipeline, AutoTokenizer, AutoModel
from datetime import datetime
from dateutil.relativedelta import relativedelta

Create raw toy clinical notes

In [3]:
clinical_notes = [
    {
        "note_id": 1,
        "note_date": "2024-01-15",
        "raw_note_text": """
        Pt is a 58 y/o male with Type 2 Diabetes.
        Currently taking Metformin 1000mg BID.
        HbA1c today is 7.8%, previously 7.2% three months ago.
        Denies chest pain.
        Plan: continue Metformin and follow up in 3 months.
        """
    },
    {
        "note_id": 2,
        "note_date": "2024-01-20",
        "raw_note_text": """
        Patient has HTN treated with Lisinopril 20mg daily.
        BP today 145/92.
        Reports dizziness last week after standing.
        Plan: monitor BP at home and return in 2 weeks.
        """
    },
    {
        "note_id": 3,
        "note_date": "2024-01-22",
        "raw_note_text": """
        Patient presents to ED with chest pain and shortness of breath.
        EKG shows ST elevation.
        Aspirin 325mg given in ED.
        Cardiology consulted immediately.
        """
    }
]

df = pd.DataFrame(clinical_notes)

df

,note_id,note_date,raw_note_text
0,1,2024-01-15,\n Pt is a 58 y/o male with Type 2 Diab...
1,2,2024-01-20,\n Patient has HTN treated with Lisinop...
2,3,2024-01-22,\n Patient presents to ED with chest pa...


Clean text

In [4]:
def clean_text(text):
    text = re.sub(r"\s+", " ", text)
    text = text.strip()
    return text


df["clean_text"] = df["raw_note_text"].apply(clean_text)

df[["note_id", "clean_text"]]

,note_id,clean_text
0,1,Pt is a 58 y/o male with Type 2 Diabetes. Curr...
1,2,Patient has HTN treated with Lisinopril 20mg d...
2,3,Patient presents to ED with chest pain and sho...


Tokenization

In [5]:
def tokenize_text(text):
    tokens = re.findall(r"[A-Za-z0-9/%\.]+", text)
    return tokens


df["tokens"] = df["clean_text"].apply(tokenize_text)

df[["note_id", "tokens"]]

,note_id,tokens
0,1,"[Pt, is, a, 58, y/o, male, with, Type, 2, Diab..."
1,2,"[Patient, has, HTN, treated, with, Lisinopril,..."
2,3,"[Patient, presents, to, ED, with, chest, pain,..."


Normalize abbreviations

In [6]:
abbreviation_map = {
    "pt": "patient",
    "htn": "hypertension",
    "bp": "blood pressure",
    "ekg": "electrocardiogram",
    "sob": "shortness of breath",
    "bid": "twice daily",
    "ed": "emergency department",
    "hba1c": "hemoglobin a1c"
}


def normalize_text(text):
    normalized = text.lower()

    for abbreviation, full_form in abbreviation_map.items():
        pattern = r"\b" + abbreviation + r"\b"
        normalized = re.sub(pattern, full_form, normalized)

    normalized = re.sub(r"\s+", " ", normalized).strip()

    return normalized


df["normalized_text"] = df["clean_text"].apply(normalize_text)

df[["note_id", "normalized_text"]]

,note_id,normalized_text
0,1,patient is a 58 y/o male with type 2 diabetes....
1,2,patient has hypertension treated with lisinopr...
2,3,patient presents to emergency department with ...


Load biomedical NER model

In [7]:
ner_model = pipeline(
    task="token-classification",
    model="d4data/biomedical-ner-all",
    aggregation_strategy="simple"
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/266M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/373 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

Extract medical entities

In [8]:
def extract_entities(text):
    raw_entities = ner_model(text)

    entities = []

    for entity in raw_entities:
        entities.append({
            "text": entity["word"],
            "label": entity["entity_group"],
            "confidence": round(float(entity["score"]), 2)
        })

    return entities


df["entities"] = df["clean_text"].apply(extract_entities)

df[["note_id", "entities"]]

,note_id,entities
0,1,"[{'text': 'pt', 'label': 'Disease_disorder', '..."
1,2,"[{'text': 'htn', 'label': 'Medication', 'confi..."
2,3,"[{'text': 'presents', 'label': 'Clinical_event..."


Print entities clearly

In [9]:
for i in range(len(df)):
    print("NOTE ID:", df.iloc[i]["note_id"])
    print("NOTE:", df.iloc[i]["clean_text"])
    print("\nENTITIES FOUND:")

    for entity in df.iloc[i]["entities"]:
        print("-", entity["text"], "|", entity["label"], "| confidence:", entity["confidence"])

    print("=" * 80)

NOTE ID: 1
NOTE: Pt is a 58 y/o male with Type 2 Diabetes. Currently taking Metformin 1000mg BID. HbA1c today is 7.8%, previously 7.2% three months ago. Denies chest pain. Plan: continue Metformin and follow up in 3 months.

ENTITIES FOUND:
- pt | Disease_disorder | confidence: 0.98
- 58 y / o | Lab_value | confidence: 1.0
- male | Sex | confidence: 0.99
- type 2 | History | confidence: 0.99
- metform | Medication | confidence: 0.95
- 1000mg | Dosage | confidence: 0.78
- hba1c | Medication | confidence: 0.98
- 7 | Lab_value | confidence: 0.95
- 7. 2 % | Lab_value | confidence: 0.89
- three months | Duration | confidence: 0.61
- chest | Biological_structure | confidence: 1.0
- metform | Medication | confidence: 0.92
NOTE ID: 2
NOTE: Patient has HTN treated with Lisinopril 20mg daily. BP today 145/92. Reports dizziness last week after standing. Plan: monitor BP at home and return in 2 weeks.

ENTITIES FOUND:
- htn | Medication | confidence: 0.74
- li | Medication | confidence: 1.0
- ##si

Context and time understanding

In [10]:
def normalize_relative_time(time_text, note_date):
    note_datetime = datetime.strptime(note_date, "%Y-%m-%d")
    time_text = time_text.lower()

    if time_text == "today":
        return note_datetime.strftime("%Y-%m-%d")

    if time_text in ["three months ago", "3 months ago"]:
        return (note_datetime - relativedelta(months=3)).strftime("%Y-%m-%d")

    if time_text == "last week":
        return (note_datetime - relativedelta(weeks=1)).strftime("%Y-%m-%d")

    if time_text == "in 3 months":
        return (note_datetime + relativedelta(months=3)).strftime("%Y-%m-%d")

    if time_text == "in 2 weeks":
        return (note_datetime + relativedelta(weeks=2)).strftime("%Y-%m-%d")

    if time_text == "immediately":
        return note_datetime.strftime("%Y-%m-%d") + " same day"

    return "not stated"


def extract_time_expressions(text, note_date):
    possible_times = [
        "today",
        "three months ago",
        "3 months ago",
        "last week",
        "in 3 months",
        "in 2 weeks",
        "immediately"
    ]

    results = []
    text_lower = text.lower()

    for time_text in possible_times:
        if time_text in text_lower:
            results.append({
                "time_text": time_text,
                "normalized_time": normalize_relative_time(time_text, note_date)
            })

    return results


df["time_expressions"] = df.apply(
    lambda row: extract_time_expressions(row["clean_text"], row["note_date"]),
    axis=1
)

df[["note_id", "time_expressions"]]

,note_id,time_expressions
0,1,"[{'time_text': 'today', 'normalized_time': '20..."
1,2,"[{'time_text': 'today', 'normalized_time': '20..."
2,3,"[{'time_text': 'immediately', 'normalized_time..."


Load ClinicalBERT

This adds ClinicalBERT to the system.

ClinicalBERT will help compare the meaning of a note with the meaning of candidate code descriptions.

In [15]:
clinicalbert_model_name = "emilyalsentzer/Bio_ClinicalBERT"

clinicalbert_tokenizer = AutoTokenizer.from_pretrained(clinicalbert_model_name)
clinicalbert_model = AutoModel.from_pretrained(clinicalbert_model_name)

config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/436M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/436M [00:00<?, ?B/s]

BertModel LOAD REPORT from: emilyalsentzer/Bio_ClinicalBERT
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Create ClinicalBERT embeddings

In [14]:
def get_clinicalbert_embedding(text):
    inputs = clinicalbert_tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=512
    )

    with torch.no_grad():
        outputs = clinicalbert_model(**inputs)

    cls_embedding = outputs.last_hidden_state[:, 0, :]
    embedding = cls_embedding.squeeze().numpy()

    return embedding

Test ClinicalBERT embedding

In [16]:
sample_embedding = get_clinicalbert_embedding(
    "Patient has Type 2 Diabetes and takes Metformin."
)

print("ClinicalBERT embedding created.")
print("Embedding length:", len(sample_embedding))

ClinicalBERT embedding created.
Embedding length: 768


Create code vocabulary mapping

This is a toy vocabulary

It includes:

ICD-style diagnosis candidates
CPT-style procedure candidates
SNOMED-style concept names
research/reporting categories

In [17]:
clinical_code_vocabulary = [
    {
        "concept": "type_2_diabetes",
        "code_system": "ICD-10-CM",
        "code": "E11.9",
        "description": "Type 2 diabetes mellitus without complications",
        "snomed_ct_candidate": "Type 2 diabetes mellitus",
        "reporting_category": "chronic_condition",
        "research_category": "diabetes_cohort"
    },
    {
        "concept": "hypertension",
        "code_system": "ICD-10-CM",
        "code": "I10",
        "description": "Essential hypertension",
        "snomed_ct_candidate": "Essential hypertension",
        "reporting_category": "chronic_condition",
        "research_category": "hypertension_cohort"
    },
    {
        "concept": "chest_pain",
        "code_system": "ICD-10-CM",
        "code": "R07.9",
        "description": "Chest pain, unspecified",
        "snomed_ct_candidate": "Chest pain",
        "reporting_category": "symptom",
        "research_category": "cardiac_symptom_cohort"
    },
    {
        "concept": "shortness_of_breath",
        "code_system": "ICD-10-CM",
        "code": "R06.02",
        "description": "Shortness of breath",
        "snomed_ct_candidate": "Shortness of breath",
        "reporting_category": "symptom",
        "research_category": "respiratory_symptom_cohort"
    },
    {
        "concept": "abnormal_ekg",
        "code_system": "ICD-10-CM",
        "code": "R94.31",
        "description": "Abnormal electrocardiogram",
        "snomed_ct_candidate": "Abnormal electrocardiogram",
        "reporting_category": "diagnostic_finding",
        "research_category": "cardiac_testing_cohort"
    },
    {
        "concept": "ekg_service",
        "code_system": "CPT",
        "code": "CPT_EKG_CANDIDATE",
        "description": "Electrocardiogram service or interpretation candidate",
        "snomed_ct_candidate": "Electrocardiogram procedure",
        "reporting_category": "procedure_or_test",
        "research_category": "diagnostic_testing"
    },
    {
        "concept": "medication_reconciliation",
        "code_system": "Workflow",
        "code": "MED_RECON_REVIEW",
        "description": "Medication reconciliation review needed",
        "snomed_ct_candidate": "Medication review",
        "reporting_category": "care_management",
        "research_category": "medication_management"
    },
    {
        "concept": "follow_up_needed",
        "code_system": "Workflow",
        "code": "FOLLOW_UP_NEEDED",
        "description": "Follow-up visit or monitoring planned",
        "snomed_ct_candidate": "Follow-up encounter",
        "reporting_category": "care_plan",
        "research_category": "follow_up_tracking"
    }
]

Create embeddings for code descriptions

In [18]:
for code_item in clinical_code_vocabulary:
    text_for_embedding = (
        code_item["description"] + " " +
        code_item["snomed_ct_candidate"] + " " +
        code_item["reporting_category"] + " " +
        code_item["research_category"]
    )

    code_item["embedding"] = get_clinicalbert_embedding(text_for_embedding)

print("ClinicalBERT embeddings created for all code descriptions.")

ClinicalBERT embeddings created for all code descriptions.


Cosine similarity

In [19]:
def cosine_similarity(vector_a, vector_b):
    vector_a = np.array(vector_a)
    vector_b = np.array(vector_b)

    dot_product = np.dot(vector_a, vector_b)
    norm_a = np.linalg.norm(vector_a)
    norm_b = np.linalg.norm(vector_b)

    if norm_a == 0 or norm_b == 0:
        return 0

    return dot_product / (norm_a * norm_b)

Predict candidate codes with ClinicalBERT

In [20]:
def predict_codes_with_clinicalbert(note_text, top_n=5):
    note_embedding = get_clinicalbert_embedding(note_text)

    predictions = []

    for code_item in clinical_code_vocabulary:
        similarity = cosine_similarity(
            note_embedding,
            code_item["embedding"]
        )

        predictions.append({
            "source": "clinicalbert_similarity",
            "concept": code_item["concept"],
            "code_system": code_item["code_system"],
            "candidate_code": code_item["code"],
            "description": code_item["description"],
            "snomed_ct_candidate": code_item["snomed_ct_candidate"],
            "reporting_category": code_item["reporting_category"],
            "research_category": code_item["research_category"],
            "clinicalbert_similarity": round(float(similarity), 3)
        })

    predictions = sorted(
        predictions,
        key=lambda x: x["clinicalbert_similarity"],
        reverse=True
    )

    return predictions[:top_n]

Add confidence and review logic

In [21]:
def add_confidence_and_review(predictions):
    final_predictions = []

    for prediction in predictions:
        score = prediction["clinicalbert_similarity"]

        if score >= 0.80:
            confidence = "high"
            review_decision = "review_optional_for_demo"
        elif score >= 0.65:
            confidence = "medium"
            review_decision = "human_review_recommended"
        else:
            confidence = "low"
            review_decision = "human_review_required"

        prediction["confidence"] = confidence
        prediction["review_decision"] = review_decision

        final_predictions.append(prediction)

    return final_predictions

Rule-based concept classifier

This helps catch exact evidence.

ClinicalBERT gives semantic similarity.
Rules give clear evidence.

In [22]:
def classify_concepts_with_rules(text):
    text_lower = text.lower()

    concepts = []

    if "type 2 diabetes" in text_lower or "diabetes" in text_lower:
        concepts.append({
            "source": "rule_based",
            "concept": "type_2_diabetes",
            "evidence": "Type 2 Diabetes" if "type 2 diabetes" in text_lower else "diabetes"
        })

    if "hypertension" in text_lower:
        concepts.append({
            "source": "rule_based",
            "concept": "hypertension",
            "evidence": "hypertension"
        })

    if "chest pain" in text_lower and "denies chest pain" not in text_lower:
        concepts.append({
            "source": "rule_based",
            "concept": "chest_pain",
            "evidence": "chest pain"
        })

    if "shortness of breath" in text_lower:
        concepts.append({
            "source": "rule_based",
            "concept": "shortness_of_breath",
            "evidence": "shortness of breath"
        })

    if "st elevation" in text_lower or "electrocardiogram shows" in text_lower:
        concepts.append({
            "source": "rule_based",
            "concept": "abnormal_ekg",
            "evidence": "ST elevation" if "st elevation" in text_lower else "electrocardiogram shows"
        })

    if "electrocardiogram" in text_lower:
        concepts.append({
            "source": "rule_based",
            "concept": "ekg_service",
            "evidence": "EKG" if "ekg" in text_lower else "electrocardiogram"
        })

    if "metformin" in text_lower or "lisinopril" in text_lower or "aspirin" in text_lower:
        concepts.append({
            "source": "rule_based",
            "concept": "medication_reconciliation",
            "evidence": "medication mentioned"
        })

    if "follow up" in text_lower or "return in" in text_lower:
        concepts.append({
            "source": "rule_based",
            "concept": "follow_up_needed",
            "evidence": "follow up" if "follow up" in text_lower else "return in"
        })

    return concepts

Map rule concepts to vocabularyMap rule concepts to vocabulary

In [23]:
def find_code_item_by_concept(concept_name):
    for item in clinical_code_vocabulary:
        if item["concept"] == concept_name:
            return item

    return None


def map_rule_concepts_to_codes(rule_concepts):
    mapped = []

    for concept in rule_concepts:
        code_item = find_code_item_by_concept(concept["concept"])

        if code_item is not None:
            mapped.append({
                "source": "rule_based_mapping",
                "concept": concept["concept"],
                "code_system": code_item["code_system"],
                "candidate_code": code_item["code"],
                "description": code_item["description"],
                "snomed_ct_candidate": code_item["snomed_ct_candidate"],
                "reporting_category": code_item["reporting_category"],
                "research_category": code_item["research_category"],
                "evidence": concept["evidence"],
                "confidence": "rule_match",
                "review_decision": "human_review_recommended"
            })

    return mapped

Validate evidence

In [24]:
def validate_evidence(note_text, code_candidates):
    note_lower = note_text.lower()

    validated = []

    for candidate in code_candidates:
        candidate = candidate.copy()

        evidence = candidate.get("evidence", "")

        if evidence and evidence.lower() in note_lower:
            candidate["evidence_found_in_note"] = True
            candidate["validation_reason"] = "Evidence found in note."
        elif candidate["source"] == "clinicalbert_similarity":
            candidate["evidence_found_in_note"] = False
            candidate["validation_reason"] = "ClinicalBERT semantic match. Exact evidence needs human review."
            candidate["review_decision"] = "human_review_required"
        else:
            candidate["evidence_found_in_note"] = False
            candidate["validation_reason"] = "Evidence not found exactly. Human review required."
            candidate["review_decision"] = "human_review_required"

        validated.append(candidate)

    return validated

Combine rule-based and ClinicalBERT predictions

In [25]:
def combine_predictions(rule_predictions, clinicalbert_predictions):
    combined = []

    for item in rule_predictions:
        combined.append(item)

    for item in clinicalbert_predictions:
        combined.append(item)

    return combined

Full pipeline function

In [26]:
def clinical_note_to_reporting_research_codes(raw_note_text, note_date):
    clean = clean_text(raw_note_text)
    tokens = tokenize_text(clean)
    normalized = normalize_text(clean)

    entities = extract_entities(clean)

    time_expressions = extract_time_expressions(clean, note_date)

    rule_concepts = classify_concepts_with_rules(normalized)
    rule_code_candidates = map_rule_concepts_to_codes(rule_concepts)

    bert_predictions = predict_codes_with_clinicalbert(clean, top_n=5)
    bert_predictions = add_confidence_and_review(bert_predictions)

    combined_candidates = combine_predictions(
        rule_code_candidates,
        bert_predictions
    )

    validated_candidates = validate_evidence(
        clean,
        combined_candidates
    )

    result = {
        "note_date": note_date,
        "clean_text": clean,
        "tokens": tokens,
        "normalized_text": normalized,
        "entities": entities,
        "time_expressions": time_expressions,
        "rule_concepts": rule_concepts,
        "validated_code_candidates": validated_candidates
    }

    return result

Run the pipeline on all notes

In [27]:
pipeline_results = []

for i in range(len(df)):
    result = clinical_note_to_reporting_research_codes(
        raw_note_text=df.iloc[i]["raw_note_text"],
        note_date=df.iloc[i]["note_date"]
    )

    pipeline_results.append(result)

df["pipeline_result"] = pipeline_results

Print final output

In [28]:
for i in range(len(df)):
    result = df.iloc[i]["pipeline_result"]

    print("NOTE ID:", df.iloc[i]["note_id"])
    print("NOTE DATE:", result["note_date"])
    print()

    print("CLEAN TEXT:")
    print(result["clean_text"])
    print()

    print("NORMALIZED TEXT:")
    print(result["normalized_text"])
    print()

    print("TIME EXPRESSIONS:")
    for time_item in result["time_expressions"]:
        print("-", time_item["time_text"], "→", time_item["normalized_time"])
    print()

    print("ENTITIES:")
    for entity in result["entities"]:
        print("-", entity["text"], "|", entity["label"], "|", entity["confidence"])
    print()

    print("FINAL CANDIDATE CODES FOR REPORTING / RESEARCH:")

    for candidate in result["validated_code_candidates"]:
        print()
        print("Source:", candidate["source"])
        print("Concept:", candidate["concept"])
        print("Code system:", candidate["code_system"])
        print("Candidate code:", candidate["candidate_code"])
        print("Description:", candidate["description"])
        print("SNOMED CT candidate:", candidate["snomed_ct_candidate"])
        print("Reporting category:", candidate["reporting_category"])
        print("Research category:", candidate["research_category"])
        print("Evidence:", candidate.get("evidence", "semantic match"))
        print("Confidence:", candidate["confidence"])
        print("Review decision:", candidate["review_decision"])
        print("Evidence found in note:", candidate["evidence_found_in_note"])
        print("Validation:", candidate["validation_reason"])

    print("=" * 100)

NOTE ID: 1
NOTE DATE: 2024-01-15

CLEAN TEXT:
Pt is a 58 y/o male with Type 2 Diabetes. Currently taking Metformin 1000mg BID. HbA1c today is 7.8%, previously 7.2% three months ago. Denies chest pain. Plan: continue Metformin and follow up in 3 months.

NORMALIZED TEXT:
patient is a 58 y/o male with type 2 diabetes. currently taking metformin 1000mg twice daily. hemoglobin a1c today is 7.8%, previously 7.2% three months ago. denies chest pain. plan: continue metformin and follow up in 3 months.

TIME EXPRESSIONS:
- today → 2024-01-15
- three months ago → 2023-10-15
- in 3 months → 2024-04-15

ENTITIES:
- pt | Disease_disorder | 0.98
- 58 y / o | Lab_value | 1.0
- male | Sex | 0.99
- type 2 | History | 0.99
- metform | Medication | 0.95
- 1000mg | Dosage | 0.78
- hba1c | Medication | 0.98
- 7 | Lab_value | 0.95
- 7. 2 % | Lab_value | 0.89
- three months | Duration | 0.61
- chest | Biological_structure | 1.0
- metform | Medication | 0.92

FINAL CANDIDATE CODES FOR REPORTING / RESEARCH:



Create final table

In [29]:
final_rows = []

for i in range(len(df)):
    result = df.iloc[i]["pipeline_result"]

    for candidate in result["validated_code_candidates"]:
        final_rows.append({
            "note_id": df.iloc[i]["note_id"],
            "note_date": result["note_date"],
            "source": candidate["source"],
            "concept": candidate["concept"],
            "code_system": candidate["code_system"],
            "candidate_code": candidate["candidate_code"],
            "description": candidate["description"],
            "snomed_ct_candidate": candidate["snomed_ct_candidate"],
            "reporting_category": candidate["reporting_category"],
            "research_category": candidate["research_category"],
            "confidence": candidate["confidence"],
            "review_decision": candidate["review_decision"],
            "evidence_found_in_note": candidate["evidence_found_in_note"],
            "validation_reason": candidate["validation_reason"]
        })

final_output_table = pd.DataFrame(final_rows)

final_output_table

,note_id,note_date,source,concept,code_system,candidate_code,description,snomed_ct_candidate,reporting_category,research_category,confidence,review_decision,evidence_found_in_note,validation_reason
0,1,2024-01-15,rule_based_mapping,type_2_diabetes,ICD-10-CM,E11.9,Type 2 diabetes mellitus without complications,Type 2 diabetes mellitus,chronic_condition,diabetes_cohort,rule_match,human_review_recommended,True,Evidence found in note.
1,1,2024-01-15,rule_based_mapping,medication_reconciliation,Workflow,MED_RECON_REVIEW,Medication reconciliation review needed,Medication review,care_management,medication_management,rule_match,human_review_required,False,Evidence not found exactly. Human review requi...
2,1,2024-01-15,rule_based_mapping,follow_up_needed,Workflow,FOLLOW_UP_NEEDED,Follow-up visit or monitoring planned,Follow-up encounter,care_plan,follow_up_tracking,rule_match,human_review_recommended,True,Evidence found in note.
3,1,2024-01-15,clinicalbert_similarity,medication_reconciliation,Workflow,MED_RECON_REVIEW,Medication reconciliation review needed,Medication review,care_management,medication_management,medium,human_review_required,False,ClinicalBERT semantic match. Exact evidence ne...
4,1,2024-01-15,clinicalbert_similarity,follow_up_needed,Workflow,FOLLOW_UP_NEEDED,Follow-up visit or monitoring planned,Follow-up encounter,care_plan,follow_up_tracking,medium,human_review_required,False,ClinicalBERT semantic match. Exact evidence ne...
5,1,2024-01-15,clinicalbert_similarity,ekg_service,CPT,CPT_EKG_CANDIDATE,Electrocardiogram service or interpretation ca...,Electrocardiogram procedure,procedure_or_test,diagnostic_testing,medium,human_review_required,False,ClinicalBERT semantic match. Exact evidence ne...
6,1,2024-01-15,clinicalbert_similarity,shortness_of_breath,ICD-10-CM,R06.02,Shortness of breath,Shortness of breath,symptom,respiratory_symptom_cohort,medium,human_review_required,False,ClinicalBERT semantic match. Exact evidence ne...
7,1,2024-01-15,clinicalbert_similarity,abnormal_ekg,ICD-10-CM,R94.31,Abnormal electrocardiogram,Abnormal electrocardiogram,diagnostic_finding,cardiac_testing_cohort,low,human_review_required,False,ClinicalBERT semantic match. Exact evidence ne...
8,2,2024-01-20,rule_based_mapping,hypertension,ICD-10-CM,I10,Essential hypertension,Essential hypertension,chronic_condition,hypertension_cohort,rule_match,human_review_required,False,Evidence not found exactly. Human review requi...
9,2,2024-01-20,rule_based_mapping,medication_reconciliation,Workflow,MED_RECON_REVIEW,Medication reconciliation review needed,Medication review,care_management,medication_management,rule_match,human_review_required,False,Evidence not found exactly. Human review requi...


Save result as CSV

In [30]:
final_output_table.to_csv(
    "clinical_reporting_research_code_candidates.csv",
    index=False
)

print("CSV saved: clinical_reporting_research_code_candidates.csv")

CSV saved: clinical_reporting_research_code_candidates.csv
